<center><h1 style="margin-bottom: 0px;">Machine Learning I (25/26)</h1></center>
<center><h2 style="margin-top: 0px;">K-Nearest Neighbour Modification and Performance Benchmarking</h2></center>

#### <center> Filipe Zheng (202406753), Maiara Guedes (202407694), Sílvia Pinto (202405988) </center> <br>

In [21]:
import os
import math
import shutil
from pandas import read_csv
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

### **1. Algorithm Selection and Implementation** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">Having the freedom to choose any of the algorithms we learned in the course lectures, we decided to work with K-Nearest Neighbour. Due to its simplicity, it gives us many options for modification and improvement, and allows us to employ meticulous attention to each detail of this project.
<p style="text-indent: 2em;">To implement K-NN (K-Nearest Neighbour), we must first comprehend how it works. When K-NN receives a new object to classify, it compares it to all the objects in the training set, which are already correctly classified. This comparison is done by measuring the distance between them: how many attributes do they have in common, and what is the difference between the attributes in which they diverge? Then, it chooses the k closest training objects ("neighbours") and checks each of their target classes. The most common class in those k neighbours is the class it will assign to the new object.
<p style="text-indent: 2em;">The following code shows our implementation. It defines a class which allows us to create and use many individual KNN classifiers - each of them can be "fed" a training set, upon which it will make predictions on new, unseen datasets. It is the unmodified, conventional version of the K-NN algorithm, which we explained above.</div>

In [22]:
class KNN_Classifier():
    def __init__(self,k=5,*,distance_function=None):                                               # Initializes a new classifier.
        self.k = k                                                                                 # k is the number of neighbours to consider.
        self.df = None                                                                             # The classifier has not yet been "fed" the data - it is empty.
        self.df_target = None                                                                      # The same goes for the target dataframe.
        self.distF = distance_function or KNN_Classifier.distance                                  # Function to calculate the distance.

    def fit(self,data_frame,data_frame_target):                                                    # Fits the dataframes into the classifier.
        self.df = data_frame.reset_index(drop=True)
        self.df_target = data_frame_target.reset_index(drop=True)

    def distance(row1,row2):                                                                       # Default distance function (can be replaced when initializing).
        Sum = 0.0
        for i,j in zip(row1,row2):
            if (isinstance(i,(float,int)) and not(math.isnan(i) or math.isnan(j))): Sum+=abs(i-j)  # For each numeric attribute, add the difference to the sum.
            else: Sum += i!=j                                                                      # For each categorical attribute, add 1 if different, else 0.
        return Sum                                                                                 # Return the distance.

    def predict_row(self, row):                                                                    # Function to predict a new, unclassified object.
        dists = [(i, self.distF(row, row0)) for i, row0 in enumerate(self.df.values)]              # Calculate all distances for each of the training set objects.
        knn = [self.df_target.iloc[i] for i, _ in sorted(dists, key=lambda x: x[1])[:self.k]]      # Select the targets of the 3 closest neighbours to our object.
        counter = {}
        for i in knn:
            if not i in counter:
                counter[i] = 0                                                                     # If the class has not been seen yet, add it the group.
            counter[i] += 1                                                                        # Add one vote to the current neighbour's class.
        return max(counter, key=counter.get)                                                       # Return the predicted value for our object. by majority voting.

    def predict(self,df):                                                                          # Function to predict the target class.
        return [self.predict_row(row) for _,row in df.iterrows()]                                  # Returns target predictions for each of the rows in the dataframe.

### **2. Performance Measurement - Before Modification** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">Now, we will test this original K-NN implementation on the datasets. By measuring various statistics of its performance, we can then decide what kind of modifications we will apply on the algorithm in order to improve those measurements.
<p style="text-indent: 2em;">But first... being provided with 3 different types of data characteristics, we must reflect on which will be the most challenging and interesting to analyse for K-NN. Each of the dataset groups is oriented to a different type of "problematic" data. Respectively, groups 1 , 2 and 3 are significantly affected by noise/outliers, class imbalance and multiclass classification. Considering K-NN's algorithm, we can make the following remarks:

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **1.** *Because K-NN is a "lazy learner" - it does not have a strong mathematical foundation, since it simply relies on the proximity of data points - for small values of k, it can be very sensitive to noise and outliers (such as a few elements of one class inside a large agglomeration of elements of another class). A few outliers can also greatly affect the distribution of the data when normalizing.*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**2.** *Class imbalance is definitely a major influence on K-NN. Since its classification relies on majority voting, if a class is under-represented, it might be chosen much less than it should - because the neighbourhood of our object is statistically more likely to be dominated by the most common class.*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**3.** *Finally, multiclass classification is also troublesome. With an increase in the number of classes, the decision boundaries become more and more complex and fragmented. Also, areas where many classes overlap become very difficult to classify, since small changes to k will probably make our prediction jump from class to class.*


<p style="text-indent: 2em;">Taking into account the previous points, it seems most likely to us that dataset group 2 will impose a greater challenge to our algorithm (but all of them will definitely have some level of impact on K-NN). A very disproportionate class imbalance might not cause a loss of accuracy; maybe it will become even higher, but that is simply because practically every new object is falling into the larger class, completely disregarding the minority class.
<p style="text-indent: 2em;">Let us test our hypothesis in a tangible way: we will now use our KNN Classifier in each dataset group and analyse various performance measures, to determine which data characteristic truly causes the most damage to our predictions.</div>


In [ ]:
path = Path('class_imbalance')                                                                     # Selects the path of the folder that contains all datasets.

folder = Path('testing_class_imbalance')                                                           # Selects the path of the folder that will contain the test set.
if os.path.exists(folder):
    shutil.rmtree(folder)                                                                          # If folder exists from previous iterations, remove its content.
os.makedirs(folder)

class File():                                                                                      # Defines a dataset file.
    def __init__(self, file):                                               
        self.file = file                                                
        self.df = read_csv(file, na_values=['None', 'nan', '']) 
        self.imbalance = self.get_imbalance()                                                      # Calculates imbalance of file, to be used as a selection criteria.                                                 
        self.size = len(self.df)                                                                   # Same thing for the number of rows in the file,
        self.feature_count = len(self.df.columns)                                                  # and the number of predictive attributes.
    
    def get_imbalance(self):
        bin1 = self.df.iloc[0, -1]                                                                 # Selects the class of the first element to be used as reference.
        count1 = 0
        for value in self.df.iloc[:, -1]:                                                          # Compares every class to that same reference,
            if value == bin1: count1 += 1                                                          # and adds to count if they are equal.
        len_df = len(self.df)
        percentage = count1/len_df                                                                 # Now we can know the proportion of imbalance between our two classes.
        if (percentage < 0.5): percentage = (len_df - count1)/len_df                               # We use the proportion of the majority class.
        return percentage

files_list = []
for file in path.iterdir():
    if file.is_file(): files_list += [File(file)]                                                  # Creates a list of all files in the dataset group

test_selection = []                                                                                # From that list, we will only select a few

for i, func in [([0,1,2,3,-4,-3,-2,-1], lambda x: x.imbalance), ([0,1,2,3], lambda x: x.feature_count), ([0,1,2,3], lambda x: x.size)]:
    files_list.sort(key = func)                                                                    # Using imbalance, number of features and number of lines as the criteria...
    for item in i:
        if files_list[item].file.name not in test_selection:
            test_selection += [files_list[item].file.name]                                         # Selects a few files to be added to our test set.
#NOTE: We use the 4 files with most imbalance, the 4 files with least imbalance, the 4 files with less features and the 4 files with less lines.
# The last two criteria are created to reduce execution time during our testing phases.

for file in test_selection:
    source = 'class_imbalance/' + file
    destination = 'testing_class_imbalance/' + file
    shutil.copy(source, destination)                                                               # Copies the selected files to the test set folder.

In [ ]:
def process_dataset(file):

    df = read_csv(file, na_values=['None', 'nan', ''])

    """nulls_limit = 0.50     # Se tiver mais de 50% de nulos, a coluna é descartada
    total_lines = len(df)

    for col in list(df.columns):
        empty = df[col].isnull().sum()   # Quantidade de espaços vazios numa coluna

        if empty/total_lines > nulls_limit:
            df = df.drop(col, axis=1)    # Apaga a coluna
    """

    X = df.iloc[:, :-1]         # Divisão de dados e classes
    Y = df.iloc[:, -1]

    if Y.dtype == 'object':                  # Encoding da target (se for categórico)
        Y = Y.astype('category').cat.codes

    #X = X.apply(pd.to_numeric, errors='coerce')

    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.1,random_state=0)    # Split Treino/Teste

    for col in X.columns:                   # Encoding no conjunto de treino
         if X[col].dtype == 'object' or str(X[col].dtype) == 'category':
            continue  # Não mudamos os categóricos
            """if X[col].isnull().sum() > 0:      # Se a coluna for texto, preenchemos os buracos com a moda
                moda = X[col].mode()[0]
                X[col] = X[col].fillna(moda)
            """
            X[col] = X[col].astype('category').cat.codes
            X.loc[X[col]==-1] = float('nan')
         else:
                """ # Transformar para uma distribuição normal
                # calcular média e std ignorando NaN
                mean = np.nanmean(X_train[col], axis=0)
                std = np.nanstd(X_train[col], axis=0)
                if std==0:std=1
                # aplicar standardization
                X_train[col] = (X_train[col] - mean) / std
                mean = np.nanmean(X_test[col], axis=0)
                std = np.nanstd(X_test[col], axis=0)
                if std==0:std=1
                X_test[col] = (X_test[col] - mean) / std
                """
                #Min Max Scaler
                if np.all(np.isnan(X_train[col])): continue
                Xmax = np.nanmax(X_train[col],axis=0)
                Xmin = np.nanmin(X_train[col],axis=0)
                X_train[col] = (X_train[col] - Xmin) / (Xmax-Xmin)
                X_test[col] = (X_test[col] - Xmin) / (Xmax-Xmin)

    """imputer = SimpleImputer(strategy='mean')            # Preencher valores em falta com a média
    X_train_imputed = imputer.fit_transform(X_train)
    X_test_imputed = imputer.transform(X_test)


    scaler = StandardScaler()   # Escalonamento
    X_train = scaler.fit_transform(X_train_imputed)
    X_test = scaler.transform(X_test_imputed)
    """
    #X_train = X_train.astype(float)
    #X_test = X_test.astype(float)

    return X_train, X_test, y_train, y_test


In [25]:
path = Path('testing_class_imbalance')

accuracies = []
f1_scores = []
for f in path.iterdir():
    if f.is_file():
        X_train, X_test, y_train, y_test = process_dataset(f)

        modelo = KNN_Classifier(k=7)
        modelo.fit(X_train, y_train)
        prevision = modelo.predict(X_test)

        accuracy = accuracy_score(y_test, prevision)
        F1_score = f1_score(y_test, prevision)
        accuracies.append(accuracy)
        f1_scores.append(F1_score)
        print(f"Dataset:{f}  Accuracy: {accuracy * 100:.2f}%   F1 score:{F1_score*100:.2f}%")

c:\Users\USER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Dataset:testing_class_imbalance\dataset_1013_analcatdata_challenger.csv  Accuracy: 100.00%   F1 score:0.00%
Dataset:testing_class_imbalance\dataset_1014_analcatdata_dmft.csv  Accuracy: 82.50%   F1 score:0.00%
Dataset:testing_class_imbalance\dataset_1056_mc1.csv  Accuracy: 99.68%   F1 score:57.14%
Dataset:testing_class_imbalance\dataset_1061_ar4.csv  Accuracy: 63.64%   F1 score:33.33%
Dataset:testing_class_imbalance\dataset_1064_ar6.csv  Accuracy: 81.82%   F1 score:0.00%
Dataset:testing_class_imbalance\dataset_316_yeast_ml8.csv  Accuracy: 99.17%   F1 score:0.00%
Dataset:testing_class_imbalance\dataset_865_analcatdata_neavote.csv  Accuracy: 90.00%   F1 score:0.00%
Dataset:testing_class_imbalance\dataset_867_visualizing_livestock.csv  Accuracy: 76.92%   F1 score:85.71%
Dataset:testing_class_imbalance\dataset_875_analcatdata_chlamydia.csv  Accuracy: 80.00%   F1 score:88.89%
Dataset:testing_class_imbalance\dataset_950_arsenic-female-lung.csv  Accuracy: 98.21%   F1 score:99.10%
Dataset:testi

In [26]:
#noise_outliers
path = "noise_outliers"
files_names = os.listdir(path)
files_noise = [os.path.join(path, name) for name in files_names]
accuracies_noise = []
f1_scores_noise = []
for f in files_noise:files_names = os.listdir(path)

    X_train, X_test, y_train, y_test = process_dataset(f)

    # Treinar o KNN
    modelo = KNN_Classifier(k=7)
    modelo.fit(X_train, y_train)


    prevision = modelo.predict(X_test)

    # Calcular a Accuracy
    accuracy = accuracy_score(y_test, prevision)
    F1_score = f1_score(y_test, prevision)
    accuracies_noise.append(accuracy)
    f1_scores_noise.append(F1_score)
    print(f"Dataset:{f}  Accuracy: {accuracy * 100:.2f}%   F1 Score: {F1_score*100:.2f}%")

IndentationError: unexpected indent (2042108923.py, line 9)

In [ ]:
print("Class Imbalance:")
print(f"Size: {len(accuracies)}")
print(f"Mean Accuracy: {np.mean(accuracies)*100:.2f}%\nMean F1 Score:{np.mean(f1_scores)*100:.2f}%")
print("\nNoise/Outliers:")
print(f"Size: {len(accuracies_noise)}")
print(f"Mean Accuracy: {np.mean(accuracies_noise)*100:.2f}%\nMean F1 Score:{np.mean(f1_scores_noise)*100:.2f}%")

Class Imbalance:
Size: 50
Mean Accuracy: 93.24%
Mean F1 Score:65.54%

Noise/Outliers:
Size: 50
Mean Accuracy: 82.73%
Mean F1 Score:79.24%


In [ ]:
for file in path.iterdir():
    df_check = read_csv(file, na_values=['None'])

    null_counts = df_check.isnull().sum()

    missing_data = null_counts[null_counts > 0]

    print("Quantidade de valores em falta por coluna:")
    print(missing_data)

    print(f"\nTotal de valores em falta no dataset: {df_check.isnull().sum().sum()}")

Quantidade de valores em falta por coluna:
age       1
sex     150
TSH     369
T3      769
TT4     231
T4U     387
FTI     385
TBG    3772
dtype: int64

Total de valores em falta no dataset: 6064
Quantidade de valores em falta por coluna:
Series([], dtype: int64)

Total de valores em falta no dataset: 0
Quantidade de valores em falta por coluna:
Series([], dtype: int64)

Total de valores em falta no dataset: 0
Quantidade de valores em falta por coluna:
Series([], dtype: int64)

Total de valores em falta no dataset: 0
Quantidade de valores em falta por coluna:
Series([], dtype: int64)

Total de valores em falta no dataset: 0
Quantidade de valores em falta por coluna:
Series([], dtype: int64)

Total de valores em falta no dataset: 0
Quantidade de valores em falta por coluna:
correl      24
cancor2    240
fract2     240
dtype: int64

Total de valores em falta no dataset: 504
Quantidade de valores em falta por coluna:
Series([], dtype: int64)

Total de valores em falta no dataset: 0
Quanti